In [1]:
#In this notebook, I want to construct the stiffness matrix for a helical spring and compute the effective spring that describes its motion.
#I will follow Steve's method to create the force constant matrix for rigid bodies from the one acting on point particles.
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

#Construct matrix r_x such that r_x v = r x v.
def cross_matrix(r):
    return np.array([[    0, -r[2],  r[1]],
                     [ r[2],     0, -r[0]],
                     [-r[1],  r[0],     0]])

#Create matrix that changes point mass force constants to rigid body force constants.
def A(r):
    return np.block([np.eye(3), -cross_matrix(r)])

#Create force constant matrix for simple springs connecting point masses.
def Phi(r1, r2, kL, kT):
    n = (r1-r2)/np.linalg.norm(r1-r2)
    phi = kL*np.outer(n,n)+kT*(np.eye(3)-np.outer(n,n))
    return phi

#Add up the right on and off-diagonal blocks that a spring contributes.
def pair_hessian(xA, xB, rA, rB, kL, kT):
    """
    xA, xB : equilibrium positions of the two contact points in world coordinates
    rA, rB : vectors from the respective rigid-body reference points
             to the contact points
    """

    phi = Phi(xA, xB, kL, kT)

    AA = A(rA)
    AB = A(rB)

    HAA = AA.T @ phi @ AA
    HAB = -AA.T @ phi @ AB
    HBA = -AB.T @ phi @ AA
    HBB = AB.T @ phi @ AB

    return np.block([
        [HAA, HAB],
        [HBA, HBB]
    ])

#Construct matrix B that takes one from 12D infinitesimal motion coordinates to 6D relative pose coordinates.
def relative_pose_matrix(d):
    I = np.eye(3)
    Z = np.zeros((3, 3))
    X = cross_matrix(d)

    return np.block([[-I,  X,  I,  Z],
                     [ Z, -I,  Z,  I]])


def relative_stiffness(H, d):
    B = relative_pose_matrix(d)
    Bplus = np.linalg.pinv(B)

    Krel = Bplus.T @ H @ Bplus

    return Krel

#Geometry of the problem: two unit cubes centered at (0,0,0) and (0,0,2).
#Connect their vertices in such a way to make helical spring.
#examine the structure of the 6x6 matrix to understand what's going on.

#centers of cubes
RA = np.array([0., 0., 0.])
RB = np.array([0., 0., 2.])

#positions of connectors relative to cube centers
a1 = np.array([ 0.5,  0.5, 0.5])
a2 = np.array([-0.5,  0.5, 0.5])
a3 = np.array([-0.5, -0.5, 0.5])
a4 = np.array([ 0.5, -0.5, 0.5])

b1 = np.array([ 0.5,  0.5, -0.5])
b2 = np.array([-0.5,  0.5, -0.5])
b3 = np.array([-0.5, -0.5, -0.5])
b4 = np.array([ 0.5, -0.5, -0.5])

#lab frame positions of spring connections
A_points = [RA + a for a in [a1, a2, a3, a4]]
B_points = [RB + b for b in [b1, b2, b3, b4]]

#make left- and right-handed helical springs: make 12x12 matrix of derivatives.
kL = 1.0
#adding kT violates SE(3) invariance.
kT = 0.0

Hr = pair_hessian(A_points[0], B_points[1], a1, b2, kL, kT) + pair_hessian(A_points[1], B_points[2], a2, b3, kL, kT) + pair_hessian(A_points[2], B_points[3], a3, b4, kL, kT) + pair_hessian(A_points[3], B_points[0], a4, b1, kL, kT) + pair_hessian(A_points[0], B_points[0], a1, b1, kL, kT) + pair_hessian(A_points[1], B_points[1], a2, b2, kL, kT) + pair_hessian(A_points[2], B_points[2], a3, b3, kL, kT) + pair_hessian(A_points[3], B_points[3], a4, b4, kL, kT)
Hl = pair_hessian(A_points[0], B_points[3], a1, b4, kL, kT) + pair_hessian(A_points[1], B_points[0], a2, b1, kL, kT) + pair_hessian(A_points[2], B_points[1], a3, b2, kL, kT) + pair_hessian(A_points[3], B_points[2], a4, b3, kL, kT) + pair_hessian(A_points[0], B_points[0], a1, b1, kL, kT) + pair_hessian(A_points[1], B_points[1], a2, b2, kL, kT) + pair_hessian(A_points[2], B_points[2], a3, b3, kL, kT) + pair_hessian(A_points[3], B_points[3], a4, b4, kL, kT)

In [2]:
B = relative_pose_matrix(np.array([0., 0., 2.]))

Krel_r = relative_stiffness(Hr, np.array([0., 0., 2.]))
Krel_l = relative_stiffness(Hl, np.array([0., 0., 2.]))

Hr_reconstructed = B.T @ Krel_r @ B
Hl_reconstructed = B.T @ Krel_l @ B

#The Hessians are reconstructable as long as kT=0.
errorr = np.linalg.norm(Hr - Hr_reconstructed)
print(errorr)

errorl = np.linalg.norm(Hl - Hl_reconstructed)
print(errorl)

7.658455818615837e-15
7.658455818615837e-15


In [3]:
pd.DataFrame(Krel_r).round(2)

,0,1,2,3,4,5
0,1.0,-0.0,0.0,-0.50,-1.00,0.0
1,-0.0,1.0,0.0,1.00,-0.50,0.0
2,0.0,0.0,6.0,0.00,0.00,1.0
3,-0.5,1.0,0.0,2.25,0.00,-0.0
4,-1.0,-0.5,0.0,0.00,2.25,-0.0
5,0.0,0.0,1.0,-0.00,-0.00,0.5


In [4]:
pd.DataFrame(Krel_l).round(2)

,0,1,2,3,4,5
0,1.0,0.0,0.0,0.50,-1.00,0.0
1,0.0,1.0,0.0,1.00,0.50,-0.0
2,0.0,0.0,6.0,0.00,0.00,-1.0
3,0.5,1.0,0.0,2.25,-0.00,0.0
4,-1.0,0.5,0.0,-0.00,2.25,-0.0
5,0.0,-0.0,-1.0,0.00,-0.00,0.5


In [5]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from scipy.linalg import expm


def cross_matrix(r):
    return np.array([
        [0,    -r[2],  r[1]],
        [r[2],  0,    -r[0]],
        [-r[1], r[0],  0]
    ])


def relative_pose_matrix(d):
    I = np.eye(3)
    Z = np.zeros((3, 3))
    X = cross_matrix(d)

    return np.block([
        [-I,  X,  I,  Z],
        [ Z, -I,  Z,  I]
    ])


def animate_relative_coordinate(d, coordinate,
                                amplitude=0.25,
                                nframes=60,
                                interval=40):

    """
    Animate one of the six canonical relative-pose coordinates.

    coordinate:
        0 = relative x translation
        1 = relative y translation
        2 = relative z translation
        3 = relative x rotation
        4 = relative y rotation
        5 = relative z rotation
    """

    # ------------------------------------------------------------
    # Relative-pose map
    # ------------------------------------------------------------

    B = relative_pose_matrix(d)
    Bplus = np.linalg.pinv(B)

    e = np.zeros(6)
    e[coordinate] = 1.0

    # 12-dimensional rigid-body motion corresponding to
    # one unit of the chosen relative coordinate.
    q = Bplus @ e

    uA = q[0:3]
    thA = q[3:6]

    uB = q[6:9]
    thB = q[9:12]

    # ------------------------------------------------------------
    # Cube geometry
    # ------------------------------------------------------------

    # Centers
    xA = np.array([0., 0., 0.])
    xB = np.array(d, dtype=float)

    # Vertices relative to each cube's center
    vertices = np.array([
        [ 0.5,  0.5,  0.5],
        [-0.5,  0.5,  0.5],
        [-0.5, -0.5,  0.5],
        [ 0.5, -0.5,  0.5],
        [ 0.5,  0.5, -0.5],
        [-0.5,  0.5, -0.5],
        [-0.5, -0.5, -0.5],
        [ 0.5, -0.5, -0.5]
    ])

    # This only draws the four vertices you used for the springs.
    # Add the other four cube vertices if desired.
    edges = [
        (0, 1),
        (1, 2),
        (2, 3),
        (3, 0),
        (0, 4),
        (1, 5),
        (2, 6),
        (3, 7),
        (4, 5),
        (5, 6),
        (6, 7),
        (7, 4)
    ]

    bonds1 = [
        (0, 5),
        (1, 6),
        (2, 7),
        (3, 4)
    ]
    bonds2 = [
        (0, 4),
        (1, 5),
        (2, 6),
        (3, 7)
    ]

    # ------------------------------------------------------------
    # Figure
    # ------------------------------------------------------------

    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(111, projection="3d")

    names = [
        "relative x translation",
        "relative y translation",
        "relative z translation",
        "relative x rotation",
        "relative y rotation",
        "relative z rotation",
    ]

    def draw_cube(V):
        for i, j in edges:
            ax.plot(
                [V[i, 0], V[j, 0]],
                [V[i, 1], V[j, 1]],
                [V[i, 2], V[j, 2]],
                linewidth=2, c = 'b'
            )
            
    def draw_bonds1(VA,VB):
        for i, j in bonds1:
            ax.plot(
                [VA[i, 0], VB[j, 0]],
                [VA[i, 1], VB[j, 1]],
                [VA[i, 2], VB[j, 2]],
                linewidth=2, c = 'r', zorder =  -1
            )

    def draw_bonds2(VA,VB):
        for i, j in bonds2:
            ax.plot(
                [VA[i, 0], VB[j, 0]],
                [VA[i, 1], VB[j, 1]],
                [VA[i, 2], VB[j, 2]],
                linewidth=2, c = 'orange', zorder =  -1
            )

    def update(frame):

        # Smooth oscillation between -amplitude and +amplitude
        s = amplitude * np.sin(
            2*np.pi * frame / nframes
        )

        # Rotate the cubes as rigid bodies
        RA = expm(s * cross_matrix(thA))
        RB = expm(s * cross_matrix(thB))

        VA = xA + s*uA + (RA @ vertices.T).T
        VB = xB + s*uB + (RB @ vertices.T).T

        ax.clear()

        draw_cube(VA)
        draw_cube(VB)
        draw_bonds1(VA,VB)
        draw_bonds2(VA,VB)

        # Centers
        centerA = xA + s*uA
        centerB = xB + s*uB

        ax.scatter(
            *centerA,
            s=40
        )

        ax.scatter(
            *centerB,
            s=40
        )

        # --------------------------------------------------------
        # Plot formatting
        # --------------------------------------------------------

        ax.set_xlim(-1.5, 1.5)
        ax.set_ylim(-1.5, 1.5)

        # Adjust automatically for the separation of the cubes
        zmin = min(-1.0, d[2] - 1.0)
        zmax = max( 1.0, d[2] + 1.0)

        ax.set_zlim(zmin, zmax)

        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.set_zlabel("z")

        ax.set_title(names[coordinate])

        # Equal aspect ratio
        ax.set_box_aspect([
            3,
            3,
            max(3, zmax-zmin)
        ])

    ani = FuncAnimation(
        fig,
        update,
        frames=nframes,
        interval=interval,
        blit=False
    )

    plt.close(fig)

    return HTML(ani.to_jshtml())

In [6]:
animate_relative_coordinate(
    d=np.array([0., 0., 2.]),
    coordinate=5,
    amplitude=0.5
)

In [9]:
#construct K with both left- and right-handed bonds.
Hb = pair_hessian(A_points[0], B_points[1], a1, b2, kL, kT) + pair_hessian(A_points[1], B_points[2], a2, b3, kL, kT) + pair_hessian(A_points[2], B_points[3], a3, b4, kL, kT) + pair_hessian(A_points[3], B_points[0], a4, b1, kL, kT) + pair_hessian(A_points[0], B_points[0], a1, b1, kL, kT) + pair_hessian(A_points[1], B_points[1], a2, b2, kL, kT) + pair_hessian(A_points[2], B_points[2], a3, b3, kL, kT) + pair_hessian(A_points[3], B_points[3], a4, b4, kL, kT) + pair_hessian(A_points[0], B_points[3], a1, b4, kL, kT) + pair_hessian(A_points[1], B_points[0], a2, b1, kL, kT) + pair_hessian(A_points[2], B_points[1], a3, b2, kL, kT) + pair_hessian(A_points[3], B_points[2], a4, b3, kL, kT)
B = relative_pose_matrix(np.array([0., 0., 2.]))
Krel_b = relative_stiffness(Hb, np.array([0., 0., 2.]))
Hb_reconstructed = B.T @ Krel_b @ B
#The Hessians are reconstructable as long as kT=0.
errorb = np.linalg.norm(Hb - Hb_reconstructed)
print(errorb)

1.1138323214990832e-14


In [11]:
#achiral since only antisymmetric part of off-diagonal block exists, which can be eliminated by an origin shift.
pd.DataFrame(Krel_b).round(2)

,0,1,2,3,4,5
0,2.0,0.0,-0.0,0.0,-2.0,0.0
1,0.0,2.0,0.0,2.0,0.0,0.0
2,-0.0,0.0,8.0,-0.0,0.0,-0.0
3,0.0,2.0,-0.0,3.5,-0.0,-0.0
4,-2.0,0.0,0.0,-0.0,3.5,-0.0
5,0.0,0.0,-0.0,-0.0,-0.0,1.0
